# HW3 Unit 4 — LLM Agent System (RAG + Multi-agent + Skills)

**Due Date:** 4/23, 11:59pm   

## Goals
In this homework, you will build a **multi-hop question-answering agent system** from scratch, then re-implement it with LangGraph, then extend it using the new **Agent Skills** open standard ([agentskills.io](https://agentskills.io), published December 2025, now supported by Cursor, VS Code, GitHub Copilot, OpenAI Codex, Gemini CLI, Goose, Claude Code, and 20+ other agent products).

**Dataset**: HotpotQA multi-hop questions — 50 dev-set questions, fixed seed.
**Metric**: Exact Match (EM), using the official HotpotQA normalization (lowercase, strip articles and punctuation, then compare).

---

## Part 0: Setup

Install dependencies and get a free Groq API key from [console.groq.com](https://console.groq.com) (email signup, no credit card). Optionally, also get a free OpenRouter key from [openrouter.ai](https://openrouter.ai) for the Part 3 multi-provider demo.

In [ ]:
!pip install -q groq sentence-transformers faiss-cpu datasets langgraph openai PyYAML

import os, getpass
os.environ["GROQ_API_KEY"] = getpass.getpass("Groq API key: ")
# os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OpenRouter API key (optional): ")

Load 50 HotpotQA dev questions (fixed seed) and the `BAAI/bge-small-en-v1.5` embedding model (local, CPU-only, ~130MB). (All the helper functions you'll need — including a cached Groq wrapper called `llm()` — are defined a few cells below in the "Provided helpers" cell.)

In [ ]:
from datasets import load_dataset
from sentence_transformers import SentenceTransformer

ds = load_dataset("hotpot_qa", "distractor", split="validation")
# Filter for bridge-type questions only (true multi-hop, harder for single-agent)
bridge_qs = ds.filter(lambda x: x["type"] == "bridge")
questions = bridge_qs.shuffle(seed=42).select(range(50))
embedder = SentenceTransformer("BAAI/bge-small-en-v1.5")

Look at one example to see what HotpotQA questions look like:

In [ ]:
ex = questions[0]
print("Question:", ex["question"])
print("Answer:  ", ex["answer"])
print("Type:    ", ex["type"])   # "bridge" (multi-hop) or "comparison"
print("Level:   ", ex["level"])  # "easy" / "medium" / "hard"
print(f"\nContext: {len(ex['context']['title'])} Wikipedia paragraphs")
for i in range(3):
    title = ex["context"]["title"][i]
    first_sent = ex["context"]["sentences"][i][0]
    print(f"  [{i}] {title}: {first_sent[:120]}...")
print("\nSupporting facts (ground truth):", ex["supporting_facts"])

Expected: you'll see a multi-hop question whose answer requires chaining facts across 2 of the 10 paragraphs — this is exactly the kind of question a single-shot LLM call struggles with, and which motivates the multi-agent design in Part 1.

### How the LLM is called

Throughout this homework we use **Groq**'s free API running the `llama-3.1-8b-instant` model (8B params, 131K context, 14,400 requests/day on the free tier). Under the hood, every LLM call boils down to this:

In [ ]:
import os
from groq import Groq

client = Groq(api_key=os.environ["GROQ_API_KEY"])

response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {"role": "system", "content": "You are a concise question answerer. Answer in 1-5 words."},
        {"role": "user",   "content": "Who directed the 2016 film Arrival?"},
    ],
    temperature=0,,
)
print(response.choices[0].message.content)
# → Denis Villeneuve

We use `temperature=0,` for a balance between consistency and diversity in LLM responses. The next cell wraps this pattern into a cached `llm()` helper you'll use throughout the homework.

### Provided helpers — run this cell, then use the functions

The following cell defines all the infrastructure you'll use throughout the homework: a cached `llm()` wrapper around the Groq call shown above, FAISS retrieval helpers, the official HotpotQA eval, and (for Part 3) SKILL.md parsing and a Wikipedia REST API wrapper. **Just run the cell** — you do not need to read or modify these functions.

In [ ]:
import os, json, re, string, hashlib, collections
from pathlib import Path

import faiss
import numpy as np
import requests
import yaml
from groq import Groq

# ─── Cached Groq LLM wrapper ─────────────────────────────────────────

_groq_client = Groq(api_key=os.environ["GROQ_API_KEY"])
_GROQ_MODEL  = "llama-3.1-8b-instant"

_LLM_CACHE_PATH = Path("cache/llm_responses.json")
_LLM_CACHE_PATH.parent.mkdir(exist_ok=True)
_llm_cache = (
    json.loads(_LLM_CACHE_PATH.read_text())
    if _LLM_CACHE_PATH.exists() else {}
)

def llm(system: str, user: str, temperature: float = 0.0) -> str:
    """Call Groq llama-3.1-8b-instant. Cached on (model, system, user, temperature)
    so re-running a cell is free."""
    key = hashlib.sha256(
        f"{_GROQ_MODEL}||{system}||{user}||{temperature}".encode()
    ).hexdigest()
    if key in _llm_cache:
        return _llm_cache[key]
    resp = _groq_client.chat.completions.create(
        model=_GROQ_MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user",   "content": user},
        ],
        temperature=temperature,
    )
    out = resp.choices[0].message.content
    _llm_cache[key] = out
    _LLM_CACHE_PATH.write_text(json.dumps(_llm_cache))
    return out

# ─── FAISS retrieval ─────────────────────────────────────────────────

def build_index(paragraphs):
    if not paragraphs:
        return None, []
    """Embed paragraphs with bge-small, build a FAISS index.
    Returns (index, paragraphs) — pass this tuple to retrieve()."""
    embs = embedder.encode(paragraphs, normalize_embeddings=True)
    index = faiss.IndexFlatIP(embs.shape[1])
    index.add(embs.astype(np.float32))
    return index, paragraphs

def retrieve(query, faiss_tuple, k=3):
    index, _ = faiss_tuple
    if index is None:
        return []
    """Return the top-k paragraphs most similar to `query`."""
    index, paragraphs = faiss_tuple
    q = embedder.encode([query], normalize_embeddings=True).astype(np.float32)
    _, ids = index.search(q, k)
    return [paragraphs[i] for i in ids[0]]

# ─── Official HotpotQA evaluation (Yang et al. 2018) ────────────────

def _normalize(s):
    s = s.lower()
    s = "".join(c for c in s if c not in set(string.punctuation))
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())


def _em(pred, gold):
    return float(_normalize(pred) == _normalize(gold))

def compute_em(results):
    """Average Exact Match over a list of {'prediction', 'gold'} dicts."""
    if not results:
        return 0.0
    return sum(_em(r["prediction"], r["gold"]) for r in results) / len(results)

# ─── Skill file parsing (used in Part 3) ────────────────────────────

def parse_skill_md(path):
    """Parse a SKILL.md file. Returns (frontmatter_dict, body_string)."""
    text = Path(path).read_text()
    m = re.match(r"^---\n(.*?)\n---\n(.*)$", text, re.DOTALL)
    if not m:
        raise ValueError(f"{path} is missing YAML frontmatter")
    return yaml.safe_load(m.group(1)), m.group(2).strip()

(The Wikipedia REST API helper used in Part 3 is defined next to its skill file — see §3.3.)

Quick smoke test to confirm everything is defined:

In [ ]:
print(llm("You answer in 1 word.", "What is 2 + 2?"))
# → 4
print(compute_em([{"prediction": "Paris", "gold": "paris"}]))
# → (1.0, 1.0)   # normalization lowercases before comparing

The code you **will** write in the rest of this homework: the 4 agents, the orchestrator loop, the `SkillRouter` class, and the 3 `SKILL.md` files. Everything else is provided above.

---

## Part 1: From-scratch RAG + Multi-agent loop (40 pts)

Build a 4-agent loop by hand — no framework. You will see what every agent system actually *is* underneath: a state dict, a few LLM calls, and a loop.

### 1.1 The design — 4 specialized agents on an assembly line

A single LLM call given a multi-hop question like *"What is the birth year of the director of the 2016 film Arrival?"* often fails: the model has to recall the director AND their birth year AND not hallucinate either. Our strategy is to split that one hard call into four simple, specialized ones:

In [ ]:
  question
     │
     ▼
 ┌─────────┐        ┌───────────┐       ┌──────────┐       ┌──────────┐
 │ Planner │ ─────▶ │ Retriever │ ────▶ │ Reasoner │ ────▶ │ Verifier │
 └─────────┘        └───────────┘       └──────────┘       └──────────┘
     ▲                                                          │
     │         "not_supported" (retry, up to 2 times)          │
     └──────────────────────────────────────────────────────────┘
                                                                │
                                                    "supported" │
                                                                ▼
                                                         final answer

- **Planner**: breaks the question into 2–3 simpler sub-questions
- **Retriever**: fetches the most relevant Wikipedia chunks for each sub-question
- **Reasoner**: writes a short answer from the retrieved evidence
- **Verifier**: fact-checks the answer against the evidence; if it looks unsupported, the loop goes back to the Planner for a second attempt

Concretely, on the Arrival question, a well-behaved run looks like:

| Step | Agent | Output |
|---|---|---|
| 1 | Planner | `["Who directed the 2016 film Arrival?", "What year was Denis Villeneuve born?"]` |
| 2 | Retriever | ~6 Wikipedia chunks mentioning Villeneuve, *Arrival* (2016), and 1967 |
| 3 | Reasoner | `"1967"` |
| 4 | Verifier | `"supported"` → loop exits |
| — | Final | `"1967"` |

The four agents communicate with each other by **writing into a shared Python dict called `state`**. You can think of `state` as a notebook that all 4 agents read and scribble in as the question moves down the line. §1.2 defines its exact schema.

### 1.2 The shared state

Every agent reads and writes a single Python `dict` called `state`. Here is its schema — what each key means and which agent fills it in:

In [ ]:
state = {
    "question":      str,         # the original HotpotQA question — set at init
    "paragraphs":    list[str],   # the ~10 Wikipedia paragraphs HotpotQA gives you — set at init
    "sub_questions": list[str],   # filled by Planner
    "evidence":      list[str],   # chunks the Retriever collected
    "answer":        str | None,  # filled by Reasoner
    "verdict":       str | None,  # "supported" or "not_supported", filled by Verifier
    "loop_count":    int,         # how many retry loops so far — starts at 0
}

**Where does the `state` come from?** You build one fresh `state` for each HotpotQA question, right before running the agent loop on that question. Two of its fields (`question` and `paragraphs`) are copied in from the dataset; the other five start as empty placeholders and get filled in as each agent runs.

The HotpotQA example's `context` field is nested — it's a dict with parallel lists `title` (10 titles) and `sentences` (10 lists of sentence strings). Before you build `state`, you need to flatten this into a list of 10 paragraph strings.

**Your task**: implement two small helpers.

In [ ]:
def flatten_context(context) -> list[str]:
    """Turn a HotpotQA `context` dict into a flat list of 10 paragraph strings,
    one per Wikipedia article. Each paragraph should begin with its title so
    the embedding model has a hint about the subject.

    Args:
        context: dict with two parallel lists:
            - context["title"]:     list of 10 Wikipedia article titles (str)
            - context["sentences"]: list of 10 lists of sentence strings

    Returns:
        A list of 10 strings, each shaped like "Title. sentence1 sentence2 ..."

    Example:
        Input (context):
            {
                "title": [
                    "Arrival (film)",
                    "Denis Villeneuve",
                    ...  # 8 more titles
                ],
                "sentences": [
                    [
                        "Arrival is a 2016 American science fiction film.",
                        "It was directed by Denis Villeneuve."
                    ],
                    [
                        "Denis Villeneuve is a Canadian filmmaker.",
                        "He was born on October 3, 1967."
                    ],
                    ...  # 8 more sentence lists
                ],
            }

        Output (return value):
            [
                "Arrival (film). Arrival is a 2016 American science fiction film. It was directed by Denis Villeneuve.",
                "Denis Villeneuve. Denis Villeneuve is a Canadian filmmaker. He was born on October 3, 1967.",
                ...  # 8 more strings
            ]
    """
    # TODO (hint: zip title and sentences, join each sentence list into one string,
    # and prepend the title. A list comprehension is ~3 lines.)
    pass


def init_state(ex) -> dict:
    """Build a fresh state dict for one HotpotQA example. Two fields come from
    the example; the other five are empty placeholders the agents will fill in."""
    return {
        "question":      ex["question"],
        "paragraphs":    flatten_context(ex["context"]),
        "sub_questions": [],     # Planner will fill
        "evidence":      [],     # Retriever will fill
        "answer":        None,   # Reasoner will fill
        "verdict":       None,   # Verifier will fill
        "loop_count":    0,
    }

Try it:

In [ ]:
state = init_state(questions[0])
print("Question:  ", state["question"])
print("Paragraphs:", len(state["paragraphs"]), "paragraphs")
print("First one: ", state["paragraphs"][0][:150], "...")

Expected: you'll see the question, a count of 10 paragraphs, and the first paragraph starting with its Wikipedia title. This `state` is what you'll hand to the Planner in §1.3.

### 1.3 The 4 agents

Each agent is a Python function `agent(state) -> state`. Three of them (Planner, Reasoner, Verifier) make a single LLM call using the `llm()` helper that was defined in Part 0's *Provided helpers* cell. The Retriever is pure Python — no LLM.

Reminder of the helper's signature (it's already in your notebook's global namespace after you ran the Part 0 cell — no import needed):

In [ ]:
llm(system: str, user: str, temperature: float = 0.0) -> str

Your job for each LLM-based agent is to **assemble** the call: the system and user prompts are given to you below — you just need to pass them into `llm()` and parse the response into the right field of `state`.

#### (a) Planner

Decomposes the question into 2–3 sub-questions. One `llm()` call.

In [ ]:
def planner_agent(state, retry=False):
    system = (
        "You are a question decomposer. Given a multi-hop question, break it "
        "into 2 or 3 simpler sub-questions whose answers together are sufficient "
        "to answer the original. Output ONE sub-question per line. No numbering, "
        "no preamble."
    )

    if retry and state["sub_questions"]:
        # Second attempt: tell the model the previous decomposition didn't yield
        # a supported answer, and ask for a MORE SPECIFIC one. Without this
        # branch the retry loop would just repeat the first attempt.
        prev = "\n".join(state["sub_questions"])
        user = f"""The previous decomposition did not yield a supported answer.
Produce a MORE SPECIFIC decomposition.

Original question: {state["question"]}

Previous sub-questions:
{prev}

New sub-questions:"""
    else:
        # First attempt
        user = f"""Question: {state["question"]}

Sub-questions:"""

    # TODO: call llm(system, user) and parse the response.
    #   - Split on newlines, strip whitespace / leading bullet characters
    #   - Store the resulting list in state["sub_questions"]
    return state

#### (b) Retriever

No LLM call. For each sub-question, calls the provided `retrieve(query, faiss_tuple, k=3)` helper (which embeds with bge-small and queries the per-question FAISS index) and collects the results.

In [ ]:
def retriever_agent(state, faiss_tuple):
    evidence = []
    for sq in state["sub_questions"]:
        evidence.extend(retrieve(sq, faiss_tuple, k=3))
    state["evidence"] = evidence
    return state

#### (c) Reasoner

Synthesizes the evidence into a short answer. One `llm()` call. HotpotQA answers are usually 1–5 words, so the prompts push the model to be terse.

In [ ]:
def reasoner_agent(state):
    system = (
        "You are a concise question answerer. Using ONLY the evidence below, "
        "answer the question in as few words as possible — usually 1 to 5 words. "
        "Do not add explanation. If the evidence does not support an answer, "
        "output 'unknown'."
    )
    evidence_text = "\n\n".join(state["evidence"])
    user = f"""Evidence:
{evidence_text}

Question: {state["question"]}

Answer:"""

    # TODO: call llm(system, user), strip whitespace / stray quotes,
    #       store the result in state["answer"]
    return state

#### (d) Verifier

Binary fact-check on the proposed answer. One `llm()` call. Expected output is exactly `"supported"` or `"not_supported"`.

In [ ]:
def verifier_agent(state):
    system = (
        "You are a fact-checking judge. Given a question, retrieved evidence, "
        "and a proposed answer, decide whether the answer is DIRECTLY supported "
        "by the evidence. Respond with exactly one word: 'supported' or 'not_supported'."
    )
    evidence_text = "\n\n".join(state["evidence"])
    user = f"""Question: {state["question"]}
Evidence: {evidence_text}
Proposed answer: {state["answer"]}

Verdict:"""

    # TODO: call llm(system, user), normalize the response to either
    #       "supported" or "not_supported", store in state["verdict"]
    return state

### 1.4 Running the 4 agents end-to-end on one question (no retry)

Before adding the retry loop, wire the 4 agents together in a straight line and run them on one real HotpotQA question. This tells you whether each agent works on its own, and lets you read the intermediate outputs of every stage.

Setup is given. The agent-calling cell is yours to write.

In [ ]:
# Pick one question from the 50 and build its state + FAISS index
ex = questions[0]
state = init_state(ex)
faiss_tuple = build_index(state["paragraphs"])

**Your task**: call the 4 agents in order on `state` and print a short summary after each one. Something like:

- after `planner_agent`: print the list of sub-questions
- after `retriever_agent`: print how many evidence chunks were retrieved
- after `reasoner_agent`: print the proposed answer
- after `verifier_agent`: print the verdict
- at the end: print the gold answer from `ex["answer"]` so you can eyeball whether it's right

In [ ]:
# TODO: run the 4 agents in sequence, printing the relevant field of `state`
#       after each call. Finish by printing the gold answer from ex["answer"].

Expected: you'll see 2–3 sub-questions, a handful of evidence chunks, a short proposed answer, and a `supported` / `not_supported` verdict. If the verdict is `not_supported`, the retry loop in §1.5 handles it.

### 1.5 The orchestrator loop + single-agent comparison

§1.4 ran the 4 agents in a straight line. The orchestrator adds a **retry loop**: when the Verifier rejects an answer, go back to the Planner (with `retry=True`) and try again. After `max_loops` retries, give up and return whatever answer the Reasoner produced last.

**Your task**: write `run_agent_loop(ex, max_loops=2)` that returns the final `state` dict. Concretely, the function needs to:

1. Initialize `state` from the HotpotQA example using `init_state(ex)` (from §1.2)
2. Build the FAISS index once with `faiss_tuple = build_index(state["paragraphs"])`
3. Call `planner_agent(state)` once to produce the initial sub-questions
4. Enter a `while` loop bounded by `state["loop_count"] <= max_loops`:
   a. Call `retriever_agent(state, faiss_tuple)`
   b. Call `reasoner_agent(state)`
   c. Call `verifier_agent(state)`
   d. If `state["verdict"] == "supported"`, `break` out of the loop
   e. Otherwise increment `state["loop_count"]` by 1
   f. If we still have retries left (`state["loop_count"] <= max_loops`), call `planner_agent(state, retry=True)` to refine the sub-questions before the next iteration
5. Return the final `state`

In [ ]:
def run_agent_loop(ex, max_loops=2):
    # TODO: follow steps 1-5 above
    pass

Quick check: run it on 3 sample questions and print the final `state["answer"]` + `state["loop_count"]` alongside each gold answer. Questions where `loop_count > 0` are the ones where the Verifier caught a bad first attempt — those are the most interesting to inspect.

**Single-agent baseline to compare against.** Before running the full multi-agent loop on all 50 questions, you will also run a **single-agent baseline** — the same question answered by just ONE LLM call, with all 10 HotpotQA paragraphs stuffed into the prompt. No Planner, no Retriever, no Reasoner, no Verifier, no loop. This is the "what if I just asked the LLM directly?" version.

In [ ]:
def single_agent_baseline(ex):
    system = "You are a concise question answerer. Answer in 1-5 words. No explanation."
    paragraphs = flatten_context(ex["context"])
    context = "\n\n".join(paragraphs)
    user = f"""Answer the question using ONLY the paragraphs below.

Paragraphs:
{context}

Question: {ex["question"]}

Answer:"""
    return llm(system, user).strip()

You will compute EM for **both** the full 4-agent loop and this single-agent baseline on the same 50 questions, so the marginal value of the multi-agent design is visible in numbers. If the multi-agent version is not meaningfully better, that tells you something — inspect a few questions to understand why.

**Running both on all 50 questions.** Once `run_agent_loop` and `single_agent_baseline` work on a single example, wire them up to the provided `compute_em` helper:

`compute_em` takes a list of `{"prediction", "gold"}` dicts and returns `(em, f1)` averaged across all items. Use this same helper in Part 2, Part 3, and Part 4 whenever you need to score a batch of predictions.

### Your Task (Part 1)

1. Implement the 4 agents and the orchestrator loop.
2. Run the end-to-end example in §1.4 on 3 sample questions and print traces (like the table in §1.1) as a sanity check.
3. Run the **full multi-agent loop** on all 50 questions; report EM.
4. Run the **single-agent baseline** from §1.5 on the same 50 questions; report its EM side by side with the multi-agent numbers.
5. Report how many times the Verifier rejected an answer and triggered a retry.

### Analysis Questions (Part 1)

**Q1**: Looking at your results, what's the point of splitting the work into 4 separate agents? Why not just put everything into one big LLM prompt ("Here's the sample question, here are the paragraphs, think step by step and answer")? Cite the EM numbers from your multi-agent run vs the single-agent baseline in §1.5. If the gap is small, inspect 2 sample questions and explain what (if anything) the multi-agent pipeline actually contributed.

## Part 2: LangGraph refactor (15 pts)

Re-implement the *exact same* 4-agent system using LangGraph. You will see that the framework is just a nicer notation for what you already built by hand — your agent functions need **zero changes**.

### 2.1 LangGraph concepts

Four terms you'll see throughout the skeleton. Read these first — everything else in Part 2 is just these four ideas wired together.

- **Node** — a Python function with signature `(state) -> state`. Each of your Part 1 agents (Planner / Retriever / Reasoner / Verifier) will become one node. Each node reads the current `state` dict, modifies it, and returns it. The function name is what appears in your code; the **node name** (a string like `"planner"`) is what the graph uses to refer to it.

- **Graph** — a collection of nodes plus the rules for how `state` flows between them. You register a node with `graph.add_node("planner", planner_node)`, and you declare transitions with `graph.add_edge("planner", "retriever")` (always go from A to B) or `graph.add_conditional_edges("verifier", branch_fn)` (pick the next node by calling `branch_fn(state)`). A graph is a **blueprint** — it doesn't run anything on its own; it just describes the topology.

- **Compile** — `graph.compile()` turns the blueprint into an executable state machine. The returned object (conventionally called `app`) is what actually knows how to walk the graph. You compile **once** and then reuse the same `app` for every question.

- **Invoke** — `app.invoke(initial_state)` runs the state machine on one concrete input. LangGraph starts at the entry point, calls the current node, passes its return value to the next edge, and keeps going until the graph terminates (either by reaching the special `END` sentinel or by running out of edges). It returns the final `state` dict.

**Mental model**: you write the node functions → register them in a graph with edges → compile once → invoke per question.

**Why bother vs the Part 1 hand-rolled loop?** Three wins:
1. **Declarative**: the flow between agents lives in `add_edge` / `add_conditional_edges`, not buried in Python `while` / `if`.
2. **Traceability**: LangGraph (and LangSmith, its tracing companion) records every node's input/output without you adding prints.
3. **Composition**: adding a 5th agent later is `add_node` + one edge change, no control-flow surgery.

### 2.2 Mapping from Part 1 to LangGraph

| Part 1 (hand-rolled) | Part 2 (LangGraph) |
|---|---|
| `state: dict` | `AgentState(TypedDict)` — same keys |
| Each agent function | A thin `*_node(state) -> state` wrapper added via `graph.add_node(...)` |
| Straight-line sequence | `graph.add_edge(a, b)` |
| Verifier `while` loop back | `graph.add_conditional_edges("verifier", branch_fn)` |
| `return state["answer"]` | `END` terminal node |

Note: LangGraph only passes `state` to each node, so Part 1 agents that take extra arguments (`retriever_agent(state, faiss_tuple)`, `planner_agent(state, retry=...)`) need a 1-line wrapper that pulls the extras from `state` before calling your Part 1 function.

### 2.3 Skeleton to fill in

The `AgentState` TypedDict, graph construction, routing, and compile step are all given below. You only fill in the **one-line bodies** of the four `*_node` functions.

In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, List, Optional

class AgentState(TypedDict):
    question:      str
    paragraphs:    List[str]
    sub_questions: List[str]
    evidence:      List[str]
    answer:        Optional[str]
    verdict:       Optional[str]
    loop_count:    int


# ─── Node wrappers — each is a one-line call to your Part 1 agent ───

def planner_node(state: AgentState) -> AgentState:
    # TODO: call your Part 1 planner_agent.
    # Pass retry=True when state["loop_count"] > 0 (i.e. we've been here before).
    pass

def retriever_node(state: AgentState) -> AgentState:
    # TODO: build a faiss_tuple from state["paragraphs"] and call retriever_agent.
    pass

def reasoner_node(state: AgentState) -> AgentState:
    # TODO: call reasoner_agent(state) and return the result.
    pass

def verifier_node(state: AgentState) -> AgentState:
    # TODO: call verifier_agent(state). If the verdict is "not_supported",
    #       bump state["loop_count"] by 1 before returning (so the conditional
    #       edge below can decide whether to retry or give up).
    pass


# ─── Graph construction (given) ─────────────────────────────────────

graph = StateGraph(AgentState)
graph.add_node("planner",   planner_node)
graph.add_node("retriever", retriever_node)
graph.add_node("reasoner",  reasoner_node)
graph.add_node("verifier",  verifier_node)

graph.set_entry_point("planner")
graph.add_edge("planner",   "retriever")
graph.add_edge("retriever", "reasoner")
graph.add_edge("reasoner",  "verifier")

def verifier_branch(state: AgentState) -> str:
    if state["verdict"] == "supported" or state["loop_count"] >= 2:
        return END
    return "planner"  # retry via planner with refined sub-questions

graph.add_conditional_edges("verifier", verifier_branch)
app = graph.compile()


# ─── Run on one example ─────────────────────────────────────────────

ex = questions[0]
final = app.invoke(init_state(ex))
print("Answer:", final["answer"])
print("Gold:  ", ex["answer"])

### Your Task (Part 2)

1. Fill in the 4 `*_node` function bodies.
2. Run on the same 50 questions. EM should match your Part 1 result exactly (or within 1 question).
3. Count lines of code in your Part 1 orchestrator vs your Part 2 graph definition.

---

## Part 3: Skills-style dynamic capability loading (30 pts) ⭐

You will implement a mini version of the **Agent Skills open standard**. The skills you write here are, in principle, directly loadable by Cursor, VS Code, OpenAI Codex, Gemini CLI, and the other tools that support the standard.

### 3.1 How Skills work — progressive disclosure

A skill is a directory containing a `SKILL.md` file. The spec defines three levels of context loading:

- **Level 1** (~100 tokens each, always loaded): every skill's `name` + `description` frontmatter
- **Level 2** (loaded on activation): the full `SKILL.md` body (< 5000 tokens)
- **Level 3** (loaded on demand): bundled files in optional `scripts/`, `references/`, `assets/` subdirectories

A **Skill Router** inspects each question, matches it against all level-1 descriptions, loads the level-2 body of any matching skill, and injects that body into the Reasoner's system prompt.

### 3.2 SKILL.md frontmatter specification

Required fields (you must follow these exactly — see [agentskills.io/specification](https://agentskills.io/specification)):

- `name`: 1–64 chars, lowercase `[a-z0-9-]`, no leading/trailing/consecutive hyphens, **must match the parent directory name**
- `description`: 1–1024 chars, describes what the skill does **and when to use it**

Optional: `license`, `compatibility`, `metadata`, `allowed-tools`.

### 3.3 The 3 skills (all provided — read and run, don't write)

All 3 SKILL.md files and the one Python tool they call are given to you below. You do **not** write skill content in this homework — Part 3's programming work is in §3.4 (routing) and §3.5 (Reasoner integration) where you implement the progressive-disclosure mechanism. The skills themselves are provided so you have something concrete to route.

First, make sure the skill directories exist:

In [ ]:
from pathlib import Path
for name in ["wikipedia-search", "date-reasoning", "entity-disambiguation"]:
    Path(f"skills/{name}").mkdir(parents=True, exist_ok=True)

#### Skill 1 — `skills/wikipedia-search/SKILL.md`

Write the file to disk with the `%%writefile` magic:

In [ ]:
%%writefile skills/wikipedia-search/SKILL.md
---
name: wikipedia-search
description: |
  Extend retrieval to live Wikipedia via the Wikipedia REST API when local
  evidence is insufficient. ALWAYS activate when the question asks about a
  specific person's biography (birth year, death year, nationality), an
  event's date or location, or any narrow factual query about a named
  entity that might not be covered by the 10 retrieved paragraphs.
---

# When to use
Activate in ANY of these situations:
- The question asks about a person's birth year, death year, nationality, or biography
- The question asks about a specific event's date, location, or participants
- The retrieved evidence contains fewer than 2 chunks mentioning the question's core entity
- The question asks "who/what/when/where" about something narrow and specific

# Instructions to the Reasoner
When this skill is active and the retrieved evidence does not contain the
fact you need, DO NOT guess. Instead, emit EXACTLY ONE search request in
this format:

<wiki_search>short search query</wiki_search>

The orchestrator will call the Python function `wikipedia_search(query)`
(defined below) and append its results to your evidence, then rerun you.
Emit at most ONE wiki_search per call. If the evidence IS sufficient,
answer normally in 1-5 words.

Examples:
Q: "What year was Marie Curie born?"
Evidence: [paragraphs about the Curie Institute, no biographical info]
A: <wiki_search>Marie Curie birth year</wiki_search>

Q: "What is the capital of France?"
Evidence: [paragraph stating "Paris is the capital of France"]
A: Paris

The Python tool this skill calls:

In [ ]:
import requests

_wiki_cache = {}

def wikipedia_search(query: str, max_chars: int = 800) -> list[str]:
    """Query Wikipedia REST API, return up to 3 short snippets. Cached in RAM."""
    if query in _wiki_cache:
        return _wiki_cache[query]
    r = requests.get(
        "https://en.wikipedia.org/w/api.php",
        params={"action": "query", "list": "search", "srsearch": query,
                "format": "json", "srlimit": 3},
        headers={"User-Agent": "HW2-Agent-System/1.0 (educational)"},
        timeout=10,
    )
    try:
        data = r.json()
    except Exception:
        return []
    hits = data.get("query", {}).get("search", [])
    snippets = [
        f"{h['title']}: {re.sub(r'<[^>]+>', '', h.get('snippet', ''))}"[:max_chars]
        for h in hits
    ]
    _wiki_cache[query] = snippets
    return snippets

# Smoke test
print(wikipedia_search("Denis Villeneuve director")[:1])

#### Skill 2 — `skills/date-reasoning/SKILL.md`

No external tool — pure reasoning instructions.

In [ ]:
%%writefile skills/date-reasoning/SKILL.md
---
name: date-reasoning
description: |
  Apply careful step-by-step date arithmetic when the question involves
  years, centuries, ages, "older than / younger than", "before / after",
  "how many years between", or any temporal comparison. ALWAYS activate
  when the question mentions explicit years, centuries, or time-based
  comparisons, even subtle ones like "during the Victorian era".
---

# When to use
Activate whenever the question contains any of:
- Explicit years (1945, 1776, 2016, etc.)
- Century references ("19th century", "early 2000s")
- Age / temporal comparisons ("older than", "younger than", "before", "after")
- Duration queries ("how many years between", "how long after")
- Temporal ordering ("first", "last", "which came earlier")

# Instructions to the Reasoner
Before writing the final answer, extract every year, century, or date
mentioned in the question AND the evidence. Do the arithmetic step by
step internally. Then output the final answer in 1–5 words (do NOT include
the reasoning in the output). Be careful about:
- "19th century" = years 1801–1900 (not 1901–2000)
- "X years older than Y" requires subtracting birth years
- Ordinal century names ("nineteenth") vs cardinal ("19th")

Examples:
Q: "Who was born first, Einstein or Newton?"
Evidence: "Einstein was born in 1879... Newton was born in 1643..."
Reasoning (internal): 1643 < 1879, so Newton.
A: Newton

Q: "How many years after WWII ended did the Berlin Wall fall?"
Evidence: "World War II ended in 1945... The Berlin Wall fell in 1989..."
Reasoning (internal): 1989 - 1945 = 44.
A: 44 years

(This skill has no Python tool — the "action" is purely a change in how the Reasoner prompt is written.)

#### Skill 3 — `skills/entity-disambiguation/SKILL.md`

No external tool — pure reasoning instructions.

In [ ]:
%%writefile skills/entity-disambiguation/SKILL.md
---
name: entity-disambiguation
description: |
  When retrieved evidence mentions multiple entities sharing a name, or
  when the question asks about one specific entity out of several
  candidates, use context clues to pick the right one before answering.
  ALWAYS activate when the question includes a disambiguating modifier
  ("the 2016 film X", "the British X"), when the evidence contains
  multiple same-named entities, or when phrases like "the X who/with/from"
  appear.
---

# When to use
Activate whenever any of the following are true:
- The question names an entity with a commonly shared name (people, films, places)
- The evidence contains multiple entries that could match the subject
- The question includes a disambiguating modifier (year, profession, location)
- Phrases like "the X who/with/from ..." appear — these signal disambiguation

# Instructions to the Reasoner
Before answering, LIST every candidate entity the evidence mentions that
could match the subject of the question. Then use the question's
disambiguating clue (year, profession, location, genre, etc.) to pick
exactly ONE candidate. Only then produce the final 1–5 word answer.

Examples:
Q: "Who directed the 2016 film Arrival?"
Evidence: [mentions "Arrival (1996 film)", "Arrival (2016 film)", "Arrival (song, 1974)"]
Reasoning (internal): "2016 film" disambiguates → Arrival (2016 film) → director Denis Villeneuve.
A: Denis Villeneuve

Q: "What chemical element is named after the Roman messenger god?"
Evidence: [mentions Mercury the planet, Mercury the element Hg, Mercury the god]
Reasoning (internal): "chemical element named after god" → Mercury the element.
A: Mercury

(Also no Python tool.)

### 3.4 The SkillRouter — implement progressive disclosure (this is your main Part 3 task)

**Why this matters.** Skills' whole point is *lazy loading*. At startup, the router should only know each skill's `name` and `description` (level 1, ~100 tokens per skill). The full body of a skill (level 2, up to ~5000 tokens) should be loaded from disk **only after** the router has selected that skill for a specific question. A naive implementation that loads every body upfront defeats the purpose — if you had 50 skills, you'd be paying for 50 × 5000 = 250K tokens on every question even when only 1 skill is relevant.

Your `SkillRouter` must enforce this by design. The skeleton below has three methods and three TODOs, one per method:

In [ ]:
from pathlib import Path

class SkillRouter:

    def __init__(self, skills_dir):
        """Level 1 load: walk skills_dir and store ONLY each skill's
        name + description. Do NOT store the body — that is lazy-loaded
        in get_body() on demand.
        """
        self.skills_dir = Path(skills_dir)
        self.skill_metadata = []  # list of {"name": str, "description": str}

        # TODO:
        #   For each subdirectory under self.skills_dir:
        #     1. Open the SKILL.md file inside
        #     2. Call parse_skill_md() (from Part 0) to get (frontmatter, body)
        #     3. IGNORE the body — this is level 1, we don't want it yet
        #     4. Append {"name": frontmatter["name"],
        #                "description": frontmatter["description"]}
        #        to self.skill_metadata
        #   Assert frontmatter["name"] == skill_dir.name (sanity check).

    def route(self, question: str) -> list[str]:
        """Ask the LLM which skill(s) apply. Uses ONLY level-1 metadata."""
        if not self.skill_metadata:
            return []

        # TODO:
        #   1. Build a short prompt listing every skill as
        #      "- <name>: <description>"
        #   2. Call llm(system, user) — suggested system prompt below —
        #      asking the model to return a comma-separated list of
        #      skill names that apply, or the literal word "none".
        #   3. Parse the LLM output into a list of skill names.
        #   4. Filter out any name that doesn't appear in
        #      self.skill_metadata (LLMs occasionally hallucinate names).
        #   5. Return the clean list.
        #
        # Suggested system prompt:
        system = (
            "You are a skill router. Given a question and a list of "
            "skills with descriptions, decide which skills apply. Return "
            "a comma-separated list of skill names, or the literal word "
            "'none' if no skill applies."
        )

        pass  # replace with your implementation

    def get_body(self, skill_name: str) -> str:
        """Level 2 load: lazily read the body of ONE skill from disk
        (called only after route() has selected this skill)."""
        # TODO:
        #   1. Construct the path skills_dir / skill_name / "SKILL.md"
        #   2. Call parse_skill_md() on it
        #   3. Return the body (the second element of the tuple)
        pass

**Sanity check** after you implement it:

In [ ]:
router = SkillRouter("skills")
print("Loaded metadata only:", [s["name"] for s in router.skill_metadata])

# Level-1 routing — only names and descriptions are used
print(router.route("Who directed the 2016 film Arrival?"))
# Expected: something like ['entity-disambiguation'] or similar

# Level-2 on-demand body load
print(router.get_body("wikipedia-search")[:120], "...")

### 3.5 Single-agent + Skills

Instead of plugging skills into the multi-agent pipeline (which adds noise when context is already provided), we test skills on the **single-agent baseline**: same one LLM call with all 10 paragraphs, but with the router's selected skill bodies injected into the system prompt.

In [ ]:
def single_agent_with_skills(ex, router):
    """Single LLM call with all 10 paragraphs + skill bodies in prompt."""
    active = router.route(ex['question'])
    skill_ctx = ''
    if active:
        skill_ctx = '\n\n'.join(
            f'## Active skill: {name}\n{router.get_body(name)}'
            for name in active
        ) + '\n\n'
    
    system = (
        'You are a concise question answerer.\n\n'
        + skill_ctx +
        'Answer using the paragraphs below. 1-5 words, no explanation.'
    )
    paragraphs = flatten_context(ex['context'])
    context = '\n\n'.join(paragraphs)
    user = f'Paragraphs:\n{context}\n\nQuestion: {ex["question"]}\n\nAnswer:'
    return llm(system, user).strip().strip('"').strip("'")


### Complete trace — single-agent with vs without skills

The cell below picks a question where the router activates a skill, then shows the side-by-side comparison.

In [ ]:
# Find a question where skills activate
trace_ex = None
for _ex in questions:
    _active = router.route(_ex['question'])
    if _active:
        trace_ex = _ex
        break
if trace_ex is None:
    print('WARNING: router activated no skills on any question.')
    trace_ex = questions[0]

print('=' * 70)
print('QUESTION:', trace_ex['question'])
print('GOLD:', trace_ex['answer'])
print('=' * 70)

active = router.route(trace_ex['question'])
print(f'\n[SkillRouter] Active skills: {active}')
for name in active:
    body = router.get_body(name)
    print(f'  Loaded: {name} ({len(body)} chars)')
    print(f'  First line: {body.split(chr(10))[0][:80]}')

ans_without = single_agent_baseline(trace_ex)
ans_with = single_agent_with_skills(trace_ex, router)

print(f'\nWithout skills: {ans_without}')
print(f'With skills:    {ans_with}')
print(f'Gold:           {trace_ex["answer"]}')


### Your Task (Part 3)

1. Run the three `%%writefile` cells in §3.3 so the SKILL.md files exist on disk.
2. Implement the 3 `SkillRouter` method bodies in §3.4. Your implementation must respect progressive disclosure: `__init__` loads only `name` + `description`, and `get_body()` reads the body from disk.
3. Run the complete trace cell above and verify that:
   - The router activates at least one skill on the chosen question
   - The skill body is loaded (you see its char count)
   - "With skills" and "Without skills" produce different answers
4. Run `single_agent_with_skills` on all 50 questions and compare EM to the baseline without skills.

###  Questions (Part 3)

**Q2**: Open [`anthropics/skills/skills/skill-creator/SKILL.md`](https://github.com/anthropics/skills/blob/main/skills/skill-creator/SKILL.md). The document suggests "please make the skill descriptions a little bit 'pushy'". What specific problem (failure mode) is this trying to address?  and why does an LLM-based router tend to have this problem?

## Part 4: Comparison & Reflection (15 pts)

### Your Task (Part 4)

1. **Bar chart**: Present the Exact Match (EM) performance across four configurations on the same 50 questions:
   - Vanilla LLM (no RAG, no agents)
   - LLM + RAG (single call with retrieved context, no agents)
   - Part 1: hand-rolled multi-agent system
   - Part 3: multi-agent system with Skills
2. **Table**: Report, for each configuration,  the lines of code, average LLM calls per question, and average latency.
3. **Comparision** (~400 words total): Provide a comparative analysis of the four  configurations, discussing their performance, efficiency, and trade-offs.  

### Reflection Questions (Part 4)

**Q3**: Under what conditions does multi-agent orchestration lead to better outcomes, and when might a simpler approach (e.g., a single LLM or RAG system) be sufficient? Discuss the trade-offs and give concrete examples.